In [1]:
from pathlib import Path
print(Path.cwd())

/Users/young/Downloads/MDS/M5/552 communication/project/bank+marketing


In [4]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    classification_report,
)

RANDOM_STATE = 42

In [5]:
print("CWD:", Path.cwd())
print("CSV exists:", Path("data/bank-additional-full.csv").exists())

CWD: /Users/young/Downloads/MDS/M5/552 communication/project/bank+marketing
CSV exists: True


In [6]:
df = pd.read_csv("data/bank-additional-full.csv", sep=";")

print("Shape:", df.shape)
print("First 10 columns:", list(df.columns)[:10])
print("Last 10 columns:", list(df.columns)[-10:])

# Target: y in {yes,no} -> {1,0}
assert "y" in df.columns, "Target column 'y' not found."
df["y"] = (df["y"].astype(str).str.lower() == "yes").astype(int)

pos_rate = df["y"].mean()
print(f"Positive rate (y=yes): {pos_rate:.4f} ({df['y'].sum()}/{len(df)})")

df.head()

Shape: (41188, 21)
First 10 columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week']
Last 10 columns: ['campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
Positive rate (y=yes): 0.1127 (4640/41188)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0


In [7]:
def make_features(data: pd.DataFrame, include_duration: bool) -> tuple[pd.DataFrame, pd.Series]:
    d = data.copy()

    # pdays=999 means "not previously contacted"
    if "pdays" in d.columns:
        d["ever_contacted"] = (d["pdays"] != 999).astype(int)
        # Avoid treating 999 as a normal numeric value
        d.loc[d["pdays"] == 999, "pdays"] = np.nan

    # Remove duration for deployable model
    if not include_duration and "duration" in d.columns:
        d = d.drop(columns=["duration"])

    y = d.pop("y")
    return d, y


X_deploy, y = make_features(df, include_duration=False)   # deployable: NO duration
X_upper, y_upper = make_features(df, include_duration=True)  # upper bound: WITH duration

print("Deployable X shape:", X_deploy.shape)
print("Upper-bound X shape:", X_upper.shape)
print("Targets equal:", y.equals(y_upper))
print("Duration in deployable:", "duration" in X_deploy.columns)
print("Ever_contacted added:", "ever_contacted" in X_deploy.columns)

Deployable X shape: (41188, 20)
Upper-bound X shape: (41188, 21)
Targets equal: True
Duration in deployable: False
Ever_contacted added: True


In [8]:
def stratified_split(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = 0.2,
    valid_size: float = 0.2,
    seed: int = RANDOM_STATE,
) -> dict[str, tuple[pd.DataFrame, pd.Series]]:
    # Split off test
    X_trval, X_te, y_trval, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )
    # Split remaining into train/valid
    valid_frac_of_trval = valid_size / (1 - test_size)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_trval, y_trval, test_size=valid_frac_of_trval, stratify=y_trval, random_state=seed
    )
    return {"train": (X_tr, y_tr), "valid": (X_va, y_va), "test": (X_te, y_te)}


splits_deploy = stratified_split(X_deploy, y)
splits_upper = stratified_split(X_upper, y)

print("Deployable split rates:")
for name, (_, y_part) in splits_deploy.items():
    print(f"  {name}: n={len(y_part)}, pos_rate={y_part.mean():.4f}")

print("\nUpper-bound split rates:")
for name, (_, y_part) in splits_upper.items():
    print(f"  {name}: n={len(y_part)}, pos_rate={y_part.mean():.4f}")

Deployable split rates:
  train: n=24712, pos_rate=0.1127
  valid: n=8238, pos_rate=0.1126
  test: n=8238, pos_rate=0.1126

Upper-bound split rates:
  train: n=24712, pos_rate=0.1127
  valid: n=8238, pos_rate=0.1126
  test: n=8238, pos_rate=0.1126


In [9]:
def build_preprocess(X: pd.DataFrame):
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return pre, numeric_cols, categorical_cols


X_tr, y_tr = splits_deploy["train"]
pre, num_cols, cat_cols = build_preprocess(X_tr)

print(f"Numeric cols ({len(num_cols)}):", num_cols)
print(f"\nCategorical cols ({len(cat_cols)}):", cat_cols)

Numeric cols (10): ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'ever_contacted']

Categorical cols (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [10]:
# Get splits
X_tr, y_tr = splits_deploy["train"]
X_va, y_va = splits_deploy["valid"]
X_te, y_te = splits_deploy["test"]

# Rebuild preprocess from training data (best practice)
pre, num_cols, cat_cols = build_preprocess(X_tr)

# Model 1: Logistic Regression (interpretable baseline)
logit = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
)
logit_pipe = Pipeline(steps=[("pre", pre), ("model", logit)])
logit_pipe.fit(X_tr, y_tr)

# Model 2: HistGradientBoosting (strong nonlinear baseline)
hgb = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.05,
    max_iter=400,
    random_state=RANDOM_STATE,
)
hgb_pipe = Pipeline(steps=[("pre", pre), ("model", hgb)])
hgb_pipe.fit(X_tr, y_tr)

print("Training done.")

Training done.


In [11]:
from dataclasses import dataclass
import numpy as np
import pandas as pd

@dataclass
class EvalResult:
    roc_auc: float
    pr_auc: float
    f1: float
    threshold: float
    topk_table: pd.DataFrame


def topk_metrics(y_true: np.ndarray, y_score: np.ndarray, ks=(0.01, 0.02, 0.05, 0.10, 0.20)) -> pd.DataFrame:
    n = len(y_true)
    base_rate = float(y_true.mean())
    order = np.argsort(-y_score)  # descending by score

    rows = []
    for k in ks:
        m = max(1, int(round(k * n)))
        idx = order[:m]
        y_k = y_true[idx]

        precision = float(y_k.mean())
        captured = float(y_k.sum() / max(1, y_true.sum()))  # fraction of all positives captured
        lift = float(precision / base_rate) if base_rate > 0 else np.nan

        rows.append(
            {
                "k_pct": k,
                "n_called": m,
                "precision_in_topk": precision,
                "captured_positives_frac": captured,
                "lift_vs_base": lift,
            }
        )
    return pd.DataFrame(rows)


def choose_threshold_max_f1(y_true: np.ndarray, y_score: np.ndarray) -> float:
    p, r, t = precision_recall_curve(y_true, y_score)
    f1s = (2 * p[:-1] * r[:-1]) / np.maximum(1e-12, (p[:-1] + r[:-1]))
    best_idx = int(np.nanargmax(f1s))
    return float(t[best_idx])


def evaluate_binary(y_true: np.ndarray, y_score: np.ndarray) -> EvalResult:
    roc = float(roc_auc_score(y_true, y_score))
    pr = float(average_precision_score(y_true, y_score))
    thr = choose_threshold_max_f1(y_true, y_score)
    y_pred = (y_score >= thr).astype(int)
    f1 = float(f1_score(y_true, y_pred))
    tkt = topk_metrics(y_true, y_score)
    return EvalResult(roc_auc=roc, pr_auc=pr, f1=f1, threshold=thr, topk_table=tkt)

In [12]:
def print_eval(tag: str, y_true: pd.Series, y_score: np.ndarray) -> EvalResult:
    res = evaluate_binary(y_true.to_numpy(), y_score)
    print(f"\n{tag}")
    print(f"ROC-AUC={res.roc_auc:.4f} | PR-AUC={res.pr_auc:.4f} | F1={res.f1:.4f} | thr={res.threshold:.4f}")
    display(res.topk_table)
    return res

# Logistic Regression
va_score_logit = logit_pipe.predict_proba(X_va)[:, 1]
te_score_logit = logit_pipe.predict_proba(X_te)[:, 1]
res_va_logit = print_eval("DEPLOYABLE Logit - VALID", y_va, va_score_logit)
res_te_logit = print_eval("DEPLOYABLE Logit - TEST", y_te, te_score_logit)

# HistGradientBoosting
va_score_hgb = hgb_pipe.predict_proba(X_va)[:, 1]
te_score_hgb = hgb_pipe.predict_proba(X_te)[:, 1]
res_va_hgb = print_eval("DEPLOYABLE HGB - VALID", y_va, va_score_hgb)
res_te_hgb = print_eval("DEPLOYABLE HGB - TEST", y_te, te_score_hgb)


DEPLOYABLE Logit - VALID
ROC-AUC=0.7971 | PR-AUC=0.4406 | F1=0.4899 | thr=0.6623


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.743902,0.065733,6.603737
1,0.02,165,0.745455,0.132543,6.617516
2,0.05,412,0.597087,0.265086,5.300437
3,0.10,824,0.481796,0.427802,4.276979
4,0.20,1648,0.351335,0.623922,3.118855



DEPLOYABLE Logit - TEST
ROC-AUC=0.8012 | PR-AUC=0.4597 | F1=0.5100 | thr=0.6550


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.743902,0.065733,6.603737
1,0.02,165,0.745455,0.132543,6.617516
2,0.05,412,0.606796,0.269397,5.386623
3,0.10,824,0.509709,0.452586,4.524764
4,0.20,1648,0.365898,0.649784,3.248134



DEPLOYABLE HGB - VALID
ROC-AUC=0.8042 | PR-AUC=0.4681 | F1=0.5026 | thr=0.2159


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.804878,0.071121,7.145027
1,0.02,165,0.739394,0.131466,6.563715
2,0.05,412,0.628641,0.279095,5.580542
3,0.10,824,0.506068,0.449353,4.492444
4,0.20,1648,0.362257,0.643319,3.215814



DEPLOYABLE HGB - TEST
ROC-AUC=0.8108 | PR-AUC=0.4793 | F1=0.5309 | thr=0.2034


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.768293,0.067888,6.820253
1,0.02,165,0.757576,0.134698,6.725118
2,0.05,412,0.640777,0.284483,5.688274
3,0.10,824,0.526699,0.467672,4.675589
4,0.20,1648,0.370752,0.658405,3.291227


In [13]:
# Build upper-bound splits
X_tr_u, y_tr_u = splits_upper["train"]
X_va_u, y_va_u = splits_upper["valid"]
X_te_u, y_te_u = splits_upper["test"]

# Preprocess (re-fit on upper-bound training data)
pre_u, num_cols_u, cat_cols_u = build_preprocess(X_tr_u)
print(f"Upper-bound numeric={len(num_cols_u)}, categorical={len(cat_cols_u)}")

# Train one strong baseline (HGB is enough for leakage demonstration)
hgb_u = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.05,
    max_iter=400,
    random_state=RANDOM_STATE,
)
hgb_u_pipe = Pipeline(steps=[("pre", pre_u), ("model", hgb_u)])
hgb_u_pipe.fit(X_tr_u, y_tr_u)

# Evaluate
va_score_u = hgb_u_pipe.predict_proba(X_va_u)[:, 1]
te_score_u = hgb_u_pipe.predict_proba(X_te_u)[:, 1]

res_va_u = print_eval("UPPER-BOUND (with duration) HGB - VALID", y_va_u, va_score_u)
res_te_u = print_eval("UPPER-BOUND (with duration) HGB - TEST", y_te_u, te_score_u)

Upper-bound numeric=11, categorical=10

UPPER-BOUND (with duration) HGB - VALID
ROC-AUC=0.9491 | PR-AUC=0.6478 | F1=0.6570 | thr=0.2877


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.780488,0.068966,6.928511
1,0.02,165,0.781818,0.139009,6.940321
2,0.05,412,0.725728,0.322198,6.442401
3,0.10,824,0.638350,0.566810,5.666728
4,0.20,1648,0.500000,0.887931,4.438578



UPPER-BOUND (with duration) HGB - TEST
ROC-AUC=0.9528 | PR-AUC=0.6977 | F1=0.6718 | thr=0.3615


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.902439,0.079741,8.011091
1,0.02,165,0.878788,0.156250,7.801136
2,0.05,412,0.774272,0.343750,6.873331
3,0.10,824,0.673544,0.598060,5.979152
4,0.20,1648,0.503641,0.894397,4.470897


In [14]:
va_score_u = hgb_u_pipe.predict_proba(X_va_u)[:, 1]
te_score_u = hgb_u_pipe.predict_proba(X_te_u)[:, 1]

res_va_u = print_eval("UPPER-BOUND (with duration) HGB - VALID", y_va_u, va_score_u)
res_te_u = print_eval("UPPER-BOUND (with duration) HGB - TEST", y_te_u, te_score_u)


UPPER-BOUND (with duration) HGB - VALID
ROC-AUC=0.9491 | PR-AUC=0.6478 | F1=0.6570 | thr=0.2877


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.780488,0.068966,6.928511
1,0.02,165,0.781818,0.139009,6.940321
2,0.05,412,0.725728,0.322198,6.442401
3,0.10,824,0.638350,0.566810,5.666728
4,0.20,1648,0.500000,0.887931,4.438578



UPPER-BOUND (with duration) HGB - TEST
ROC-AUC=0.9528 | PR-AUC=0.6977 | F1=0.6718 | thr=0.3615


,k_pct,n_called,precision_in_topk,captured_positives_frac,lift_vs_base
0,0.01,82,0.902439,0.079741,8.011091
1,0.02,165,0.878788,0.156250,7.801136
2,0.05,412,0.774272,0.343750,6.873331
3,0.10,824,0.673544,0.598060,5.979152
4,0.20,1648,0.503641,0.894397,4.470897


In [15]:
def rate_by(col: str) -> pd.DataFrame:
    out = (
        df.groupby(col)["y"]
          .agg(subscribe_rate="mean", n="count")
          .reset_index()
          .sort_values("subscribe_rate", ascending=False)
    )
    return out

display(rate_by("month"))
display(rate_by("contact"))
display(rate_by("poutcome"))

,month,subscribe_rate,n
5,mar,0.505495,546
2,dec,0.489011,182
9,sep,0.449123,570
8,oct,0.438719,718
0,apr,0.204787,2632
1,aug,0.106021,6178
4,jun,0.105115,5318
7,nov,0.101439,4101
3,jul,0.090466,7174
6,may,0.064347,13769


,contact,subscribe_rate,n
0,cellular,0.147376,26144
1,telephone,0.052313,15044


,poutcome,subscribe_rate,n
2,success,0.651129,1373
0,failure,0.142286,4252
1,nonexistent,0.088322,35563


In [16]:
# Top-10% summary table (deployable)
top10_deploy = pd.DataFrame([
    {"model": "Deployable Logit (TEST)", "top10_precision": 0.509709, "top10_lift": 4.524764},
    {"model": "Deployable HGB (TEST)",   "top10_precision": 0.526699, "top10_lift": 4.675589},
])

top10_deploy

,model,top10_precision,top10_lift
0,Deployable Logit (TEST),0.509709,4.524764
1,Deployable HGB (TEST),0.526699,4.675589
